In [ ]:
# %% 셀 1: unique 텍스트 수집 + 임베딩 계산
import os, json, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

POS_DIR = "./data/8_telop_position"
EMB_SAVE_PATH = "./data/8_text_embeddings.pt"
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_EMB = 2048  # 임베딩 배치 크기

# ── 1) unique 텍스트 수집 ──
unique_texts = set()
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    unique_texts.add(channel)  # 채널명
    for fname in sorted(os.listdir(ch_dir)):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(ch_dir, fname), "r") as f:
            data = json.load(f)
        for inst in data.get("instances", []):
            unique_texts.add(inst["text"])  # 텔롭 텍스트

unique_texts = sorted(unique_texts)
print(f"📝 unique 텍스트: {len(unique_texts):,}개")


# ── 2) 임베딩 계산 ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_emb = AutoModel.from_pretrained(MODEL_NAME).half().cuda().eval()

text2emb = {}
for i in tqdm(range(0, len(unique_texts), BATCH_EMB), desc="임베딩"):
    batch_texts = unique_texts[i:i + BATCH_EMB]
    inputs = tokenizer(batch_texts, padding=True, truncation=True,
                       max_length=128, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_emb(**inputs)
        embs = outputs.last_hidden_state[:, 0, :].float().cpu()  # (B, 1024)

    for text, emb in zip(batch_texts, embs):
        text2emb[text] = emb

# ── 3) 저장 ──
torch.save(text2emb, EMB_SAVE_PATH)
print(f"✅ 저장: {EMB_SAVE_PATH} ({len(text2emb):,}개 텍스트)")

# 메모리 해제
del model_emb, tokenizer
torch.cuda.empty_cache()

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📝 unique 텍스트: 1,448,729개


Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
임베딩: 100%|██████████| 708/708 [11:12<00:00,  1.05it/s]


✅ 저장: ./data/8_text_embeddings.pt (1,448,729개 텍스트)
